# Test run: SAA with larger $N$ and $M$

Not part of the assignment. The procedure of §4.3 in `chapter4.ipynb` with $M = 20$ batches, larger $N$ and a validation sample of $N' = 100{,}000$ (batch seeds $7000 + 100\,\text{idx} + m$, validation seed 7999), with $U = \bar v + t_{0.975,M-1}\, s_v / \sqrt{M}$. Expected behaviour:

- $\bar v$ decreases towards $v^\star$ as $N$ grows (optimistic bias of SAA);
- $s_v \approx s_\Pi / \sqrt{N}$, so $s_v \sqrt{N}$ stays roughly constant;
- $U \ge L$, and the half-widths and the gap $U - L$ shrink.

In [1]:
%run chapter4.ipynb

mu = 353.12, sigma = 76.03


VSS = 34.86, 95% CI = [26.24, 43.48]


In [2]:
from scipy.stats import t as t_dist

TEST_SETTINGS = [(50, 20), (1000, 20), (2000, 20), (10_000, 20), (50_000, 20)]
N_VAL_TEST, VAL_SEED_TEST = 100_000, 7999
val_test = generate(N_VAL_TEST, VAL_SEED_TEST, inst)[1:]

test_candidates, test_summary = [], []
for idx, (N, M) in enumerate(TEST_SETTINGS, start=1):
    start = time.perf_counter()
    cand, solve_time, eval_time = saa_batches(N, [7000 + 100 * idx + m for m in range(1, M + 1)], val_test, inst)
    # upper bound on v*
    v_mean, v_sd = cand["v_hat"].mean(), cand["v_hat"].std(ddof=1)
    U_half = t_dist.ppf(0.975, M - 1) * v_sd / np.sqrt(M)
    U = v_mean + U_half
    # select best candidate on validation sample
    best = cand.loc[cand["g_hat"].idxmax()]
    # lower bound from the profits of the selected candidate
    L_half = 1.96 * best["sd_Pi"] / np.sqrt(N_VAL_TEST)
    L = best["g_hat"] - L_half
    cand.insert(0, "N", N)
    test_candidates.append(cand)
    test_summary.append({"N": N, "M": M, "chosen_m": int(best["m"]),
                         "active": active(best[Q_COLS].to_numpy(dtype=float)), "Q_total": best[Q_COLS].sum(),
                         "Q_total_sd": cand[Q_COLS].sum(axis=1).std(ddof=1),
                         "v_mean": v_mean, "v_sd": v_sd, "v_sd_sqrtN": v_sd * np.sqrt(N), "U_half": U_half, "U": U,
                         "g_hat": best["g_hat"], "sd_Pi": best["sd_Pi"], "L_half": L_half, "L": L, "gap": U - L,
                         "solve_time_s": solve_time, "eval_time_s": eval_time,
                         "total_time_s": time.perf_counter() - start})
test_candidates = pd.concat(test_candidates, ignore_index=True)
test_summary = pd.DataFrame(test_summary)
test_summary.round(3)

,N,M,chosen_m,active,Q_total,Q_total_sd,v_mean,v_sd,v_sd_sqrtN,U_half,U,g_hat,sd_Pi,L_half,L,gap,solve_time_s,eval_time_s,total_time_s
0,50,20,8,5,460.779,37.691,6949.539,435.265,3077.787,203.710,7153.250,6918.343,2967.741,18.394,6899.948,253.301,0.088,0.086,0.362
1,1000,20,8,5,453.409,3.697,6948.413,94.754,2996.398,44.346,6992.759,6920.348,2973.604,18.431,6901.918,90.842,0.471,0.089,1.043
2,2000,20,18,5,453.676,3.378,6936.039,62.924,2814.067,29.450,6965.489,6920.353,2973.433,18.430,6901.924,63.565,1.075,0.090,2.045
3,10000,20,19,5,453.960,1.618,6934.896,28.222,2822.214,13.208,6948.104,6920.352,2973.247,18.428,6901.924,46.181,5.349,0.077,8.659
4,50000,20,20,5,453.635,0.991,6925.874,17.803,3980.839,8.332,6934.206,6920.353,2973.459,18.430,6901.923,32.282,42.305,0.087,60.619


In [3]:
# expected behaviour (printed, not asserted)
print("U >= L for every N:     ", bool((test_summary["U"] >= test_summary["L"]).all()))
print("v_mean decreases with N:", test_summary["v_mean"].is_monotonic_decreasing)
print("v_sd decreases with N:  ", test_summary["v_sd"].is_monotonic_decreasing)
print("gap decreases with N:   ", test_summary["gap"].is_monotonic_decreasing)

U >= L for every N:      True
v_mean decreases with N: True
v_sd decreases with N:   True
gap decreases with N:    True


Same $N$ as §4.3: $M = 5$, $N' = 10{,}000$ against $M = 20$, $N' = 100{,}000$.

In [4]:
cols = ["N", "M", "g_hat", "U", "L", "gap"]
pd.concat([saa[cols], test_summary.loc[test_summary["N"] <= 2000, cols]]).sort_values(["N", "M"], ignore_index=True).round(2)

,N,M,g_hat,U,L,gap
0,50,5,6926.97,7225.25,6868.40,356.85
1,50,20,6918.34,7153.25,6899.95,253.30
2,1000,5,6926.96,6985.56,6868.39,117.17
3,1000,20,6920.35,6992.76,6901.92,90.84
4,2000,5,6926.95,7012.23,6868.39,143.84
5,2000,20,6920.35,6965.49,6901.92,63.57


## Export

Separate from the assignment results: `results/test_run_*.csv`.

In [5]:
test_candidates.round(2).to_csv("results/test_run_candidates.csv", index=False)
test_summary.round(3).to_csv("results/test_run_summary.csv", index=False)